In [1]:
import os

for name in os.listdir("/kaggle/input/datasets/zlemglsmklkaya"):
    print(name)

caai-revision-splits-new
caai-models-nb1-complete
caai-revision-splits
caai-models-cic-scenario-t
caai-models-nb3-complete


In [2]:
import os, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import joblib

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

SPLITS_ROOT = "/kaggle/input/datasets/zlemglsmklkaya/caai-revision-splits"
NB1_ROOT    = "/kaggle/input/datasets/zlemglsmklkaya/caai-models-nb1-complete"
OLD_CC_ROOT = "/kaggle/input/datasets/zlemglsmklkaya/caai-revision-splits-new"   # ae_credit_card ONLY
AE_T_ROOT   = "/kaggle/input/datasets/zlemglsmklkaya/caai-models-cic-scenario-t"  # ae_cic_scenario_T ONLY
NB3_ROOT    = "/kaggle/input/datasets/zlemglsmklkaya/caai-models-nb3-complete"
OUTPUT_DIR  = "/kaggle/working/shap_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- Data ----
cc_dir = os.path.join(SPLITS_ROOT, "credit_card")
cc_id  = pd.read_parquet(os.path.join(cc_dir, "id.parquet"))
cc_id2 = pd.read_parquet(os.path.join(cc_dir, "id2.parquet"))
cc_ood = pd.read_parquet(os.path.join(cc_dir, "ood.parquet"))
with open(os.path.join(cc_dir, "config.json")) as f:
    CC_FEATURES = json.load(f)["model_features"]
print(f"Credit card: ID={cc_id.shape}, ID2={cc_id2.shape}, OOD={cc_ood.shape}")

cic_scenarios = {}
for scen in ["scenario_T", "scenario_A"]:
    d = os.path.join(SPLITS_ROOT, "cic_ids2017", scen)
    cic_scenarios[scen] = {
        "id":  pd.read_parquet(os.path.join(d, "id.parquet")),
        "id2": pd.read_parquet(os.path.join(d, "id2.parquet")),
        "ood": pd.read_parquet(os.path.join(d, "ood.parquet")),
    }
with open(os.path.join(SPLITS_ROOT, "cic_ids2017", "config.json")) as f:
    CIC_FEATURES = json.load(f)["model_features"]
for scen, d in cic_scenarios.items():
    print(f"{scen}: ID={d['id'].shape}, ID2={d['id2'].shape}, OOD={d['ood'].shape}")

# ---- Model classes ----
class Autoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dims, latent_dim):
        super().__init__()
        enc = []; prev = input_dim
        for h in hidden_dims:
            enc += [nn.Linear(prev, h), nn.ReLU()]; prev = h
        enc += [nn.Linear(prev, latent_dim)]
        self.encoder = nn.Sequential(*enc)
        dec = []; prev = latent_dim
        for h in reversed(hidden_dims):
            dec += [nn.Linear(prev, h), nn.ReLU()]; prev = h
        dec += [nn.Linear(prev, input_dim)]
        self.decoder = nn.Sequential(*dec)
    def forward(self, x):
        return self.decoder(self.encoder(x))

class VAE(nn.Module):
    def __init__(self, input_dim, hidden_dims, latent_dim):
        super().__init__()
        enc = []; prev = input_dim
        for h in hidden_dims:
            enc += [nn.Linear(prev, h), nn.ReLU()]; prev = h
        self.encoder_body = nn.Sequential(*enc)
        self.fc_mu = nn.Linear(prev, latent_dim)
        self.fc_logvar = nn.Linear(prev, latent_dim)
        dec = []; prev = latent_dim
        for h in reversed(hidden_dims):
            dec += [nn.Linear(prev, h), nn.ReLU()]; prev = h
        dec += [nn.Linear(prev, input_dim)]
        self.decoder_body = nn.Sequential(*dec)
        self.global_logvar = nn.Parameter(torch.zeros(1))
    def encode(self, x):
        h = self.encoder_body(x)
        mu = self.fc_mu(h)
        logvar = torch.clamp(self.fc_logvar(h), min=-10, max=10)
        return mu, logvar
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std
    def decode(self, z):
        return self.decoder_body(z)
    def forward(self, x):
        mu_z, logvar_z = self.encode(x)
        z = self.reparameterize(mu_z, logvar_z)
        return self.decode(z), mu_z, logvar_z

def load_and_verify_vae_config(config_path, label):
    with open(config_path) as f:
        cfg = json.load(f)
    output_var = cfg.get("architecture", {}).get("output_variance", "")
    assert "global scalar" in output_var, f"{label}: NOT the corrected formulation (found: '{output_var}')"
    print(f"  ✓ {label}: verified corrected formulation")
    return cfg

ARCH_CC  = {"hidden_dims": [20, 14], "latent_dim": 7}
ARCH_CIC = {"hidden_dims": [48, 32], "latent_dim": 16}

print("\nLoading neural models...")
ae_cc = Autoencoder(len(CC_FEATURES), **ARCH_CC).to(DEVICE)
ae_cc.load_state_dict(torch.load(os.path.join(OLD_CC_ROOT, "ae_credit_card", "model.pt"), map_location=DEVICE))
ae_cc.eval()
print("  ✓ ae_credit_card")

load_and_verify_vae_config(os.path.join(NB1_ROOT, "vae_credit_card", "config.json"), "vae_credit_card")
vae_cc = VAE(len(CC_FEATURES), **ARCH_CC).to(DEVICE)
vae_cc.load_state_dict(torch.load(os.path.join(NB1_ROOT, "vae_credit_card", "model.pt"), map_location=DEVICE))
vae_cc.eval()

ae_T = Autoencoder(len(CIC_FEATURES), **ARCH_CIC).to(DEVICE)
ae_T.load_state_dict(torch.load(os.path.join(AE_T_ROOT, "ae_cic_scenario_T", "model.pt"), map_location=DEVICE))
ae_T.eval()
print("  ✓ ae_cic_scenario_T")

load_and_verify_vae_config(os.path.join(NB1_ROOT, "vae_cic_scenario_T", "config.json"), "vae_cic_scenario_T")
vae_T = VAE(len(CIC_FEATURES), **ARCH_CIC).to(DEVICE)
vae_T.load_state_dict(torch.load(os.path.join(NB1_ROOT, "vae_cic_scenario_T", "model.pt"), map_location=DEVICE))
vae_T.eval()

ae_A = Autoencoder(len(CIC_FEATURES), **ARCH_CIC).to(DEVICE)
ae_A.load_state_dict(torch.load(os.path.join(NB1_ROOT, "ae_cic_scenario_A", "model.pt"), map_location=DEVICE))
ae_A.eval()
print("  ✓ ae_cic_scenario_A")

load_and_verify_vae_config(os.path.join(NB1_ROOT, "vae_cic_scenario_A", "config.json"), "vae_cic_scenario_A")
vae_A = VAE(len(CIC_FEATURES), **ARCH_CIC).to(DEVICE)
vae_A.load_state_dict(torch.load(os.path.join(NB1_ROOT, "vae_cic_scenario_A", "model.pt"), map_location=DEVICE))
vae_A.eval()

print("\nLoading Isolation Forest models...")
if_cc = joblib.load(os.path.join(NB3_ROOT, "if_credit_card", "model.joblib"))
if_T  = joblib.load(os.path.join(NB3_ROOT, "if_cic_scenario_T", "model.joblib"))
if_A  = joblib.load(os.path.join(NB3_ROOT, "if_cic_scenario_A", "model.joblib"))
print("  ✓ if_credit_card, if_cic_scenario_T, if_cic_scenario_A")

print("\nAll 9 models loaded and verified.")

Device: cuda
Credit card: ID=(21731, 31), ID2=(21731, 31), OOD=(139490, 31)
scenario_T: ID=(296457, 82), ID2=(296457, 82), OOD=(1017868, 82)
scenario_A: ID=(357584, 82), ID2=(357585, 82), OOD=(615433, 82)

Loading neural models...
  ✓ ae_credit_card
  ✓ vae_credit_card: verified corrected formulation
  ✓ ae_cic_scenario_T
  ✓ vae_cic_scenario_T: verified corrected formulation
  ✓ ae_cic_scenario_A
  ✓ vae_cic_scenario_A: verified corrected formulation

Loading Isolation Forest models...
  ✓ if_credit_card, if_cic_scenario_T, if_cic_scenario_A

All 9 models loaded and verified.


In [3]:
try:
    import shap
    print("shap version:", shap.__version__)
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "shap", "--quiet"])
    import shap
    print("shap installed, version:", shap.__version__)

import time

# ---- Background sets: summarized via k-means to keep KernelSHAP's per-instance cost bounded ----
# (using the raw Train set as background would make every explanation cost scale with
# dataset size — k-means summary decouples background cost from dataset size)
def build_background(train_like_df, feature_cols, k=50, seed=SEED):
    return shap.kmeans(train_like_df[feature_cols].values, k)

# Reuse ID sets as background source (structurally same population as Train, and we
# already have them loaded — avoids reloading Train just for this)
bg_cc  = build_background(cc_id, CC_FEATURES, k=50)
bg_T   = build_background(cic_scenarios["scenario_T"]["id"], CIC_FEATURES, k=50)
bg_A   = build_background(cic_scenarios["scenario_A"]["id"], CIC_FEATURES, k=50)
print("Background sets built (k=50 each).")

# ---- Model wrapper functions: numpy in, numpy out, whole-batch forward pass ----
def ae_predict_fn(model, X_np):
    X = torch.tensor(X_np, dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        recon = model(X)
        scores = ((recon - X) ** 2).mean(dim=1)
    return scores.cpu().numpy()

def vae_predict_fn(model, X_np):
    X = torch.tensor(X_np, dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        mu_z, _ = model.encode(X)
        out_mu = model.decode(mu_z)
        logvar = torch.clamp(model.global_logvar, min=-10, max=10)
        nll = 0.5 * (logvar + ((X - out_mu) ** 2) / torch.exp(logvar) + np.log(2 * np.pi))
        scores = nll.sum(dim=1)
    return scores.cpu().numpy()

def if_predict_fn(model, X_np):
    return -model.score_samples(X_np)  # sign-flipped, same convention as NB3

print("Wrapper functions defined.")

# ---- Single-instance timing probe: Isolation Forest (expected fastest) vs AE (neural) on Credit Card ----
test_instance = cc_id[CC_FEATURES].iloc[[0]].values

t0 = time.time()
explainer_if = shap.KernelExplainer(lambda x: if_predict_fn(if_cc, x), bg_cc)
shap_vals_if = explainer_if.shap_values(test_instance, nsamples="auto")
t_if = time.time() - t0
print(f"\nIsolation Forest, 1 instance, Credit Card: {t_if:.1f}s")

t0 = time.time()
explainer_ae = shap.KernelExplainer(lambda x: ae_predict_fn(ae_cc, x), bg_cc)
shap_vals_ae = explainer_ae.shap_values(test_instance, nsamples="auto")
t_ae = time.time() - t0
print(f"Autoencoder, 1 instance, Credit Card: {t_ae:.1f}s")

shap version: 0.51.0
Background sets built (k=50 each).
Wrapper functions defined.


  0%|          | 0/1 [00:00<?, ?it/s]


Isolation Forest, 1 instance, Credit Card: 0.5s


  0%|          | 0/1 [00:00<?, ?it/s]

Autoencoder, 1 instance, Credit Card: 0.5s


In [4]:
test_instance_T = cic_scenarios["scenario_T"]["id"][CIC_FEATURES].iloc[[0]].values

t0 = time.time()
explainer_if_T = shap.KernelExplainer(lambda x: if_predict_fn(if_T, x), bg_T)
_ = explainer_if_T.shap_values(test_instance_T, nsamples="auto")
print(f"Isolation Forest, 1 instance, scenario_T: {time.time() - t0:.1f}s")

t0 = time.time()
explainer_ae_T = shap.KernelExplainer(lambda x: ae_predict_fn(ae_T, x), bg_T)
_ = explainer_ae_T.shap_values(test_instance_T, nsamples="auto")
print(f"AE, 1 instance, scenario_T: {time.time() - t0:.1f}s")

t0 = time.time()
explainer_vae_T = shap.KernelExplainer(lambda x: vae_predict_fn(vae_T, x), bg_T)
_ = explainer_vae_T.shap_values(test_instance_T, nsamples="auto")
print(f"VAE, 1 instance, scenario_T: {time.time() - t0:.1f}s")

# What's the actual nsamples count 'auto' resolved to, for each feature-count case?
print(f"\nnsamples 'auto' for {len(CC_FEATURES)} features: {2 * len(CC_FEATURES) + 2048}")
print(f"nsamples 'auto' for {len(CIC_FEATURES)} features: {2 * len(CIC_FEATURES) + 2048}")

  0%|          | 0/1 [00:00<?, ?it/s]

Isolation Forest, 1 instance, scenario_T: 0.6s


  0%|          | 0/1 [00:00<?, ?it/s]

AE, 1 instance, scenario_T: 0.1s


  0%|          | 0/1 [00:00<?, ?it/s]

VAE, 1 instance, scenario_T: 0.3s

nsamples 'auto' for 29 features: 2106
nsamples 'auto' for 67 features: 2182


In [5]:
n_probe = 20
test_batch_cc = cc_id[CC_FEATURES].iloc[:n_probe].values
test_batch_T  = cic_scenarios["scenario_T"]["id"][CIC_FEATURES].iloc[:n_probe].values

t0 = time.time()
_ = explainer_if.shap_values(test_batch_cc, nsamples="auto")
t = time.time() - t0
print(f"Isolation Forest, {n_probe} instances, Credit Card: {t:.1f}s total, {t/n_probe:.3f}s/instance")

t0 = time.time()
_ = explainer_ae.shap_values(test_batch_cc, nsamples="auto")
t = time.time() - t0
print(f"AE, {n_probe} instances, Credit Card: {t:.1f}s total, {t/n_probe:.3f}s/instance")

t0 = time.time()
_ = explainer_ae_T.shap_values(test_batch_T, nsamples="auto")
t = time.time() - t0
print(f"AE, {n_probe} instances, scenario_T: {t:.1f}s total, {t/n_probe:.3f}s/instance")

t0 = time.time()
_ = explainer_if_T.shap_values(test_batch_T, nsamples="auto")
t = time.time() - t0
print(f"Isolation Forest, {n_probe} instances, scenario_T: {t:.1f}s total, {t/n_probe:.3f}s/instance")

  0%|          | 0/20 [00:00<?, ?it/s]

Isolation Forest, 20 instances, Credit Card: 9.2s total, 0.459s/instance


  0%|          | 0/20 [00:00<?, ?it/s]

AE, 20 instances, Credit Card: 1.5s total, 0.073s/instance


  0%|          | 0/20 [00:00<?, ?it/s]

AE, 20 instances, scenario_T: 2.3s total, 0.117s/instance


  0%|          | 0/20 [00:00<?, ?it/s]

Isolation Forest, 20 instances, scenario_T: 11.8s total, 0.589s/instance


In [6]:
t0 = time.time()
_ = explainer_vae_T.shap_values(test_batch_T, nsamples="auto")
t = time.time() - t0
print(f"VAE, {n_probe} instances, scenario_T: {t:.1f}s total, {t/n_probe:.3f}s/instance")

explainer_vae_cc = shap.KernelExplainer(lambda x: vae_predict_fn(vae_cc, x), bg_cc)
t0 = time.time()
_ = explainer_vae_cc.shap_values(test_batch_cc, nsamples="auto")
t = time.time() - t0
print(f"VAE, {n_probe} instances, Credit Card: {t:.1f}s total, {t/n_probe:.3f}s/instance")

  0%|          | 0/20 [00:00<?, ?it/s]

VAE, 20 instances, scenario_T: 2.4s total, 0.119s/instance


  0%|          | 0/20 [00:00<?, ?it/s]

VAE, 20 instances, Credit Card: 1.5s total, 0.075s/instance


In [7]:
def check_local_accuracy(shap_values, X_batch, predict_fn, background_mean):
    predictions = predict_fn(X_batch)
    shap_sums = shap_values.sum(axis=1) if shap_values.ndim > 1 else shap_values.sum()
    expected = predictions - background_mean
    residual = np.abs(shap_sums - expected)
    return residual

bg_mean_if_cc = if_predict_fn(if_cc, bg_cc.data).mean()
sv_if = explainer_if.shap_values(test_batch_cc, nsamples="auto")
resid = check_local_accuracy(sv_if, test_batch_cc, lambda x: if_predict_fn(if_cc, x), bg_mean_if_cc)
print(f"IF Credit Card local-accuracy residual: max={resid.max():.6f}, median={np.median(resid):.6f}")

bg_mean_ae_cc = ae_predict_fn(ae_cc, bg_cc.data).mean()
sv_ae = explainer_ae.shap_values(test_batch_cc, nsamples="auto")
resid = check_local_accuracy(sv_ae, test_batch_cc, lambda x: ae_predict_fn(ae_cc, x), bg_mean_ae_cc)
print(f"AE Credit Card local-accuracy residual: max={resid.max():.6f}, median={np.median(resid):.6f}")

  0%|          | 0/20 [00:00<?, ?it/s]

IF Credit Card local-accuracy residual: max=0.095819, median=0.095819


  0%|          | 0/20 [00:00<?, ?it/s]

AE Credit Card local-accuracy residual: max=3.486732, median=3.486732


In [8]:
print("Shapes:")
print("  sv_if:", np.array(sv_if).shape, "ndim:", np.array(sv_if).ndim)
print("  sv_ae:", np.array(sv_ae).shape, "ndim:", np.array(sv_ae).ndim)
print("  test_batch_cc:", test_batch_cc.shape)

preds_if = if_predict_fn(if_cc, test_batch_cc)
preds_ae = ae_predict_fn(ae_cc, test_batch_cc)
print("\n  preds_if:", preds_if.shape)
print("  preds_ae:", preds_ae.shape)
print("  bg_mean_if_cc:", bg_mean_if_cc, "(scalar)")
print("  bg_mean_ae_cc:", bg_mean_ae_cc, "(scalar)")

sv_if_arr = np.array(sv_if)
sv_ae_arr = np.array(sv_ae)

# Force explicit reshape to (n_instances, n_features) regardless of any extra trailing dims
sv_if_2d = sv_if_arr.reshape(sv_if_arr.shape[0], -1)
sv_ae_2d = sv_ae_arr.reshape(sv_ae_arr.shape[0], -1)
print("\n  sv_if reshaped:", sv_if_2d.shape)
print("  sv_ae reshaped:", sv_ae_2d.shape)

shap_sums_if = sv_if_2d.sum(axis=1)
shap_sums_ae = sv_ae_2d.sum(axis=1)
print("\n  shap_sums_if:", shap_sums_if.shape, "first 5:", shap_sums_if[:5])
print("  shap_sums_ae:", shap_sums_ae.shape, "first 5:", shap_sums_ae[:5])

resid_if = np.abs(shap_sums_if - (preds_if - bg_mean_if_cc))
resid_ae = np.abs(shap_sums_ae - (preds_ae - bg_mean_ae_cc))
print(f"\nIF residuals — shape={resid_if.shape}, max={resid_if.max():.6f}, median={np.median(resid_if):.6f}")
print(f"AE residuals — shape={resid_ae.shape}, max={resid_ae.max():.6f}, median={np.median(resid_ae):.6f}")
print(f"\nFor scale — AE prediction range: min={preds_ae.min():.4f}, max={preds_ae.max():.4f}")
print(f"For scale — IF prediction range: min={preds_if.min():.4f}, max={preds_if.max():.4f}")

Shapes:
  sv_if: (20, 29) ndim: 2
  sv_ae: (20, 29) ndim: 2
  test_batch_cc: (20, 29)

  preds_if: (20,)
  preds_ae: (20,)
  bg_mean_if_cc: 0.46495584391951605 (scalar)
  bg_mean_ae_cc: 3.7634351 (scalar)

  sv_if reshaped: (20, 29)
  sv_ae reshaped: (20, 29)

  shap_sums_if: (20,) first 5: [ 0.09126396  0.02919114 -0.00355416  0.03052468  0.03269272]
  shap_sums_ae: (20,) first 5: [-0.12483152  0.04993641 -0.12423551  0.07733041 -0.05795299]

IF residuals — shape=(20,), max=0.095819, median=0.095819
AE residuals — shape=(20,), max=3.486732, median=3.486732

For scale — AE prediction range: min=0.0409, max=0.6482
For scale — IF prediction range: min=0.3442, max=0.4655


In [9]:
bg_scores_ae = ae_predict_fn(ae_cc, bg_cc.data)
print("Background (k-means centroids) AE scores:")
print(f"  min={bg_scores_ae.min():.4f}, median={np.median(bg_scores_ae):.4f}, "
      f"mean={bg_scores_ae.mean():.4f}, max={bg_scores_ae.max():.4f}")
print(f"  All 50 values: {sorted(bg_scores_ae)}")

print(f"\nFor comparison — real ID instances' AE scores (first 20):")
print(f"  min={preds_ae.min():.4f}, median={np.median(preds_ae):.4f}, mean={preds_ae.mean():.4f}")

Background (k-means centroids) AE scores:
  min=0.0059, median=0.0970, mean=3.7634, max=67.8426
  All 50 values: [np.float32(0.00594404), np.float32(0.020238737), np.float32(0.02063274), np.float32(0.022623979), np.float32(0.026098698), np.float32(0.02627043), np.float32(0.03392614), np.float32(0.03526091), np.float32(0.036575414), np.float32(0.037522126), np.float32(0.040994994), np.float32(0.04578074), np.float32(0.061347455), np.float32(0.061489865), np.float32(0.06389057), np.float32(0.064370334), np.float32(0.069702566), np.float32(0.076268755), np.float32(0.076435626), np.float32(0.07786031), np.float32(0.08571681), np.float32(0.087053575), np.float32(0.094290555), np.float32(0.094783716), np.float32(0.094934925), np.float32(0.09902852), np.float32(0.11294606), np.float32(0.12042918), np.float32(0.14620052), np.float32(0.1556432), np.float32(0.26606748), np.float32(0.28735897), np.float32(0.31086123), np.float32(0.41287667), np.float32(0.41756433), np.float32(0.43354574), np.floa

In [10]:
def build_background_from_samples(source_df, feature_cols, n=50, seed=SEED):
    rng = np.random.RandomState(seed)
    idx = rng.choice(len(source_df), size=min(n, len(source_df)), replace=False)
    return source_df[feature_cols].iloc[idx].values

bg_cc_v2 = build_background_from_samples(cc_id[cc_id["Class"] == 0], CC_FEATURES, n=50)  # benign-only, matches training distribution
bg_T_v2  = build_background_from_samples(cic_scenarios["scenario_T"]["id"][cic_scenarios["scenario_T"]["id"]["Class"] == 0], CIC_FEATURES, n=50)
bg_A_v2  = build_background_from_samples(cic_scenarios["scenario_A"]["id"][cic_scenarios["scenario_A"]["id"]["Class"] == 0], CIC_FEATURES, n=50)

bg_scores_ae_v2 = ae_predict_fn(ae_cc, bg_cc_v2)
print("Sampled-background AE scores (Credit Card):")
print(f"  min={bg_scores_ae_v2.min():.4f}, median={np.median(bg_scores_ae_v2):.4f}, "
      f"mean={bg_scores_ae_v2.mean():.4f}, max={bg_scores_ae_v2.max():.4f}")
print("(compare to real ID instances: min=0.0409, median=0.1810, mean=0.2275)")

Sampled-background AE scores (Credit Card):
  min=0.0306, median=0.1359, mean=0.2005, max=0.7291
(compare to real ID instances: min=0.0409, median=0.1810, mean=0.2275)


In [11]:
explainer_ae_v2 = shap.KernelExplainer(lambda x: ae_predict_fn(ae_cc, x), bg_cc_v2)
sv_ae_v2 = explainer_ae_v2.shap_values(test_batch_cc, nsamples="auto")

bg_mean_ae_v2 = bg_scores_ae_v2.mean()
shap_sums_ae_v2 = np.array(sv_ae_v2).sum(axis=1)
resid_ae_v2 = np.abs(shap_sums_ae_v2 - (preds_ae - bg_mean_ae_v2))

print(f"AE (sampled background) local-accuracy residuals:")
print(f"  max={resid_ae_v2.max():.6f}, median={np.median(resid_ae_v2):.6f}")
print(f"  relative to prediction scale (median pred={np.median(preds_ae):.4f}): "
      f"{np.median(resid_ae_v2)/np.median(preds_ae)*100:.2f}%")
print(f"\n  All 20 residuals: {resid_ae_v2}")

  0%|          | 0/20 [00:00<?, ?it/s]

AE (sampled background) local-accuracy residuals:
  max=0.000000, median=0.000000
  relative to prediction scale (median pred=0.1810): 0.00%

  All 20 residuals: [3.14414501e-08 3.14414501e-08 3.14414502e-08 1.17570162e-07
 1.32620335e-08 1.32620335e-08 9.08970835e-09 2.07126141e-08
 1.63912775e-09 1.65402889e-08 1.32471323e-07 1.65402889e-08
 5.79655170e-08 3.88920307e-08 4.30643558e-08 1.17570162e-07
 1.65402890e-08 1.63912775e-09 1.32620335e-08 7.61449336e-08]


In [12]:
KERNELSHAP_CONFIG = {
    "background_method": "sampled real benign rows from ID split (NOT k-means — k-means centroids can land off-manifold and corrupt the background baseline)",
    "background_size": 50,
    "nsamples": "auto",
    "n_instances_target": 200,
}
print(json.dumps(KERNELSHAP_CONFIG, indent=2))

# Verify T and A backgrounds land in a sane range too, before trusting them
bg_scores_ae_T = ae_predict_fn(ae_T, bg_T_v2)
print(f"\nScenario_T sampled-background AE scores: min={bg_scores_ae_T.min():.4f}, "
      f"median={np.median(bg_scores_ae_T):.4f}, max={bg_scores_ae_T.max():.4f}")

bg_scores_ae_A = ae_predict_fn(ae_A, bg_A_v2)
print(f"Scenario_A sampled-background AE scores: min={bg_scores_ae_A.min():.4f}, "
      f"median={np.median(bg_scores_ae_A):.4f}, max={bg_scores_ae_A.max():.4f}")

{
  "background_method": "sampled real benign rows from ID split (NOT k-means \u2014 k-means centroids can land off-manifold and corrupt the background baseline)",
  "background_size": 50,
  "nsamples": "auto",
  "n_instances_target": 200
}

Scenario_T sampled-background AE scores: min=0.0001, median=0.0011, max=0.1267
Scenario_A sampled-background AE scores: min=0.0001, median=0.0019, max=0.0275


In [13]:
def run_kernelshap_for_split(predict_fn, background, df, feature_cols, split_name, config=KERNELSHAP_CONFIG, seed=SEED):
    explainer = shap.KernelExplainer(predict_fn, background)
    results = {}
    for cls in [0, 1]:
        class_df = df[df["Class"] == cls]
        if len(class_df) == 0:
            print(f"    {split_name} Class={cls}: no instances, skipping")
            continue
        rng = np.random.RandomState(seed)
        n_sample = min(config["n_instances_target"], len(class_df))
        idx = rng.choice(len(class_df), size=n_sample, replace=False)
        X_sample = class_df[feature_cols].iloc[idx].values

        shap_vals = explainer.shap_values(X_sample, nsamples=config["nsamples"])
        shap_vals = np.array(shap_vals)

        # Local-accuracy spot-check on a subsample (full check on every instance would double runtime)
        n_check = min(10, n_sample)
        preds = np.array([predict_fn(X_sample[i:i+1])[0] for i in range(n_check)])
        bg_mean = predict_fn(background).mean()
        resid = np.abs(shap_vals[:n_check].sum(axis=1) - (preds - bg_mean))
        max_resid_pct = (resid / (np.abs(preds) + 1e-8) * 100).max()

        results[cls] = {"shap_values": shap_vals, "n_instances": n_sample, "max_local_accuracy_resid_pct": float(max_resid_pct)}
        print(f"    {split_name} Class={cls}: {n_sample} instances, "
              f"max local-accuracy residual = {max_resid_pct:.4f}% of prediction")
    return results


def run_full_kernelshap(predict_fn, background, splits_dict, feature_cols, model_label, config=KERNELSHAP_CONFIG):
    all_results = {}
    for split_name, split_df in splits_dict.items():
        print(f"  --- {model_label} — {split_name} ---")
        all_results[split_name] = run_kernelshap_for_split(predict_fn, background, split_df, feature_cols, split_name, config)
    return all_results


def save_shap_results(results_by_split, model_label, output_dir, config=KERNELSHAP_CONFIG):
    model_dir = os.path.join(output_dir, f"shap_{model_label}")
    os.makedirs(model_dir, exist_ok=True)
    summary = {}
    for split_name, split_results in results_by_split.items():
        for cls, res in split_results.items():
            key = f"{split_name}_class{cls}"
            np.save(os.path.join(model_dir, f"shap_values_{key}.npy"), res["shap_values"])
            summary[key] = {"n_instances": res["n_instances"], "max_local_accuracy_resid_pct": res["max_local_accuracy_resid_pct"]}
    with open(os.path.join(model_dir, "config.json"), "w") as f:
        json.dump({"model": model_label, "kernelshap_config": config, "summary": summary}, f, indent=2)
    print(f"Saved: {model_dir}")
    print(os.listdir(model_dir))


cc_splits_shap = {"id": cc_id, "id2": cc_id2, "ood": cc_ood}

print("========== Credit Card — all 3 models ==========")
print("AE:")
ae_cc_shap = run_full_kernelshap(lambda x: ae_predict_fn(ae_cc, x), bg_cc_v2, cc_splits_shap, CC_FEATURES, "ae_credit_card")
print("VAE:")
vae_cc_shap = run_full_kernelshap(lambda x: vae_predict_fn(vae_cc, x), bg_cc_v2, cc_splits_shap, CC_FEATURES, "vae_credit_card")
print("Isolation Forest:")
if_cc_shap = run_full_kernelshap(lambda x: if_predict_fn(if_cc, x), bg_cc_v2, cc_splits_shap, CC_FEATURES, "if_credit_card")

========== Credit Card — all 3 models ==========
AE:
  --- ae_credit_card — id ---


  0%|          | 0/200 [00:00<?, ?it/s]

    id Class=0: 200 instances, max local-accuracy residual = 0.0000% of prediction


  0%|          | 0/136 [00:00<?, ?it/s]

    id Class=1: 136 instances, max local-accuracy residual = 0.0000% of prediction
  --- ae_credit_card — id2 ---


...

    id2 Class=1: 136 instances, max local-accuracy residual = 0.0000% of prediction
  --- if_credit_card — ood ---


  0%|          | 0/200 [00:00<?, ?it/s]

    ood Class=0: 200 instances, max local-accuracy residual = 0.0000% of prediction


  0%|          | 0/200 [00:00<?, ?it/s]

    ood Class=1: 200 instances, max local-accuracy residual = 0.0000% of prediction


In [14]:
save_shap_results(ae_cc_shap, "ae_credit_card", OUTPUT_DIR)
save_shap_results(vae_cc_shap, "vae_credit_card", OUTPUT_DIR)
save_shap_results(if_cc_shap, "if_credit_card", OUTPUT_DIR)

Saved: /kaggle/working/shap_results/shap_ae_credit_card
['shap_values_id2_class0.npy', 'shap_values_ood_class0.npy', 'shap_values_id_class1.npy', 'shap_values_id_class0.npy', 'config.json', 'shap_values_id2_class1.npy', 'shap_values_ood_class1.npy']
Saved: /kaggle/working/shap_results/shap_vae_credit_card
['shap_values_id2_class0.npy', 'shap_values_ood_class0.npy', 'shap_values_id_class1.npy', 'shap_values_id_class0.npy', 'config.json', 'shap_values_id2_class1.npy', 'shap_values_ood_class1.npy']
Saved: /kaggle/working/shap_results/shap_if_credit_card
['shap_values_id2_class0.npy', 'shap_values_ood_class0.npy', 'shap_values_id_class1.npy', 'shap_values_id_class0.npy', 'config.json', 'shap_values_id2_class1.npy', 'shap_values_ood_class1.npy']


In [15]:
T_id  = cic_scenarios["scenario_T"]["id"]
T_id2 = cic_scenarios["scenario_T"]["id2"]
T_ood = cic_scenarios["scenario_T"]["ood"]
T_splits_shap = {"id": T_id, "id2": T_id2, "ood": T_ood}

print("========== scenario_T — all 3 models ==========")
print("AE:")
ae_T_shap = run_full_kernelshap(lambda x: ae_predict_fn(ae_T, x), bg_T_v2, T_splits_shap, CIC_FEATURES, "ae_cic_scenario_T")
print("VAE:")
vae_T_shap = run_full_kernelshap(lambda x: vae_predict_fn(vae_T, x), bg_T_v2, T_splits_shap, CIC_FEATURES, "vae_cic_scenario_T")
print("Isolation Forest:")
if_T_shap = run_full_kernelshap(lambda x: if_predict_fn(if_T, x), bg_T_v2, T_splits_shap, CIC_FEATURES, "if_cic_scenario_T")

========== scenario_T — all 3 models ==========
AE:
  --- ae_cic_scenario_T — id ---


  0%|          | 0/200 [00:00<?, ?it/s]

    id Class=0: 200 instances, max local-accuracy residual = 0.0004% of prediction


  0%|          | 0/200 [00:00<?, ?it/s]

    id Class=1: 200 instances, max local-accuracy residual = 0.0002% of prediction
  --- ae_cic_scenario_T — id2 ---


...

    id2 Class=1: 200 instances, max local-accuracy residual = 0.0000% of prediction
  --- if_cic_scenario_T — ood ---


  0%|          | 0/200 [00:00<?, ?it/s]

    ood Class=0: 200 instances, max local-accuracy residual = 0.0000% of prediction


  0%|          | 0/200 [00:00<?, ?it/s]

    ood Class=1: 200 instances, max local-accuracy residual = 0.0000% of prediction


In [16]:
save_shap_results(ae_T_shap, "ae_cic_scenario_T", OUTPUT_DIR)
save_shap_results(vae_T_shap, "vae_cic_scenario_T", OUTPUT_DIR)
save_shap_results(if_T_shap, "if_cic_scenario_T", OUTPUT_DIR)

Saved: /kaggle/working/shap_results/shap_ae_cic_scenario_T
['shap_values_id2_class0.npy', 'shap_values_ood_class0.npy', 'shap_values_id_class1.npy', 'shap_values_id_class0.npy', 'config.json', 'shap_values_id2_class1.npy', 'shap_values_ood_class1.npy']
Saved: /kaggle/working/shap_results/shap_vae_cic_scenario_T
['shap_values_id2_class0.npy', 'shap_values_ood_class0.npy', 'shap_values_id_class1.npy', 'shap_values_id_class0.npy', 'config.json', 'shap_values_id2_class1.npy', 'shap_values_ood_class1.npy']
Saved: /kaggle/working/shap_results/shap_if_cic_scenario_T
['shap_values_id2_class0.npy', 'shap_values_ood_class0.npy', 'shap_values_id_class1.npy', 'shap_values_id_class0.npy', 'config.json', 'shap_values_id2_class1.npy', 'shap_values_ood_class1.npy']


In [17]:
A_id  = cic_scenarios["scenario_A"]["id"]
A_id2 = cic_scenarios["scenario_A"]["id2"]
A_ood = cic_scenarios["scenario_A"]["ood"]
A_splits_shap = {"id": A_id, "id2": A_id2, "ood": A_ood}

print("========== scenario_A — all 3 models ==========")
print("AE:")
ae_A_shap = run_full_kernelshap(lambda x: ae_predict_fn(ae_A, x), bg_A_v2, A_splits_shap, CIC_FEATURES, "ae_cic_scenario_A")
print("VAE:")
vae_A_shap = run_full_kernelshap(lambda x: vae_predict_fn(vae_A, x), bg_A_v2, A_splits_shap, CIC_FEATURES, "vae_cic_scenario_A")
print("Isolation Forest:")
if_A_shap = run_full_kernelshap(lambda x: if_predict_fn(if_A, x), bg_A_v2, A_splits_shap, CIC_FEATURES, "if_cic_scenario_A")

========== scenario_A — all 3 models ==========
AE:
  --- ae_cic_scenario_A — id ---


  0%|          | 0/200 [00:00<?, ?it/s]

    id Class=0: 200 instances, max local-accuracy residual = 0.0006% of prediction


  0%|          | 0/200 [00:00<?, ?it/s]

    id Class=1: 200 instances, max local-accuracy residual = 0.0001% of prediction
  --- ae_cic_scenario_A — id2 ---


...

    id2 Class=1: 200 instances, max local-accuracy residual = 0.0000% of prediction
  --- if_cic_scenario_A — ood ---


  0%|          | 0/200 [00:00<?, ?it/s]

    ood Class=0: 200 instances, max local-accuracy residual = 0.0000% of prediction


  0%|          | 0/200 [00:00<?, ?it/s]

    ood Class=1: 200 instances, max local-accuracy residual = 0.0000% of prediction


In [18]:
print("Splits present in if_A_shap:", list(if_A_shap.keys()))
for split_name, split_results in if_A_shap.items():
    print(f"  {split_name}: classes present = {list(split_results.keys())}, "
          f"shapes = {[split_results[c]['shap_values'].shape for c in split_results]}")

Splits present in if_A_shap: ['id', 'id2', 'ood']
  id: classes present = [0, 1], shapes = [(200, 67), (200, 67)]
  id2: classes present = [0, 1], shapes = [(200, 67), (200, 67)]
  ood: classes present = [0, 1], shapes = [(200, 67), (200, 67)]


In [19]:
save_shap_results(ae_A_shap, "ae_cic_scenario_A", OUTPUT_DIR)
save_shap_results(vae_A_shap, "vae_cic_scenario_A", OUTPUT_DIR)
save_shap_results(if_A_shap, "if_cic_scenario_A", OUTPUT_DIR)

Saved: /kaggle/working/shap_results/shap_ae_cic_scenario_A
['shap_values_id2_class0.npy', 'shap_values_ood_class0.npy', 'shap_values_id_class1.npy', 'shap_values_id_class0.npy', 'config.json', 'shap_values_id2_class1.npy', 'shap_values_ood_class1.npy']
Saved: /kaggle/working/shap_results/shap_vae_cic_scenario_A
['shap_values_id2_class0.npy', 'shap_values_ood_class0.npy', 'shap_values_id_class1.npy', 'shap_values_id_class0.npy', 'config.json', 'shap_values_id2_class1.npy', 'shap_values_ood_class1.npy']
Saved: /kaggle/working/shap_results/shap_if_cic_scenario_A
['shap_values_id2_class0.npy', 'shap_values_ood_class0.npy', 'shap_values_id_class1.npy', 'shap_values_id_class0.npy', 'config.json', 'shap_values_id2_class1.npy', 'shap_values_ood_class1.npy']


In [20]:
import shutil
shutil.make_archive("/kaggle/working/shap_archive_nb5", "zip", "/kaggle/working/shap_results")
print("Size (MB):", os.path.getsize("/kaggle/working/shap_archive_nb5.zip") / (1024*1024))

Size (MB): 0.9199333190917969
